# Lab 2: Seq2Seq Neural Machine Translation
This notebook is the student skeleton for implementing a GRU-based Seq2Seq model.

In [6]:
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

# Import pre-built Seq2Seq model
from seq2seq_model import Encoder, Decoder, Seq2Seq

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cpu


## 1. Load parallel data

In [7]:
def load_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

train_src = load_lines("data/train.en") # Truy xuất tập huấn luyện nguồn (tiếng Anh)
train_tgt = load_lines("data/train.vi") # Truy xuất tập huấn luyện đích (tiếng Việt)
dev_src = load_lines("data/dev.en")
dev_tgt = load_lines("data/dev.vi")

# Kiểm tra biến
print(train_src[:3])
print(train_tgt[:3])


['i am a student', 'i am a teacher', 'he likes football']
['tôi là sinh viên', 'tôi là giáo viên', 'anh ấy thích bóng đá']


## 2. Build vocabularies
Reuse or complete the utilities from Lab 1.

In [8]:
SPECIAL_TOKENS = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]

# Lớp tạo thư viện vocab
class Vocab:
    def __init__(self):
        self.word2id = {}
        self.id2word = {}
        for token in SPECIAL_TOKENS:
            self.add_word(token)

    def add_word(self, word):
        if word not in self.word2id:
            idx = len(self.word2id)
            self.word2id[word] = idx
            self.id2word[idx] = word

    def build(self, sentences):
        """Build vocabulary from list of sentences"""
        for sentence in sentences:
            tokens = tokenize(sentence)
            for token in tokens:
                self.add_word(token)
        return self

    def encode(self, sentence):
        """Encode sentence to list of IDs with <BOS> and <EOS>"""
        tokens = tokenize(sentence)
        ids = [self.word2id["<BOS>"]]
        for token in tokens:
            ids.append(self.word2id.get(token, self.word2id["<UNK>"]))
        ids.append(self.word2id["<EOS>"])
        return ids

    def decode(self, ids):
        words = []
        for idx in ids:
            token = self.id2word.get(int(idx), "<UNK>")
            if token == "<EOS>":
                break
            if token not in ["<PAD>", "<BOS>"]:
                words.append(token)
        return " ".join(words)

#Tokenization có từ Lab 1 để tách các từ
def tokenize(sentence):
    return sentence.lower().split()


## 3. Encode data and pad mini-batches

### Ví dụ thực hiện tiền xử lý

In [9]:
# Hàm gán thêm các đệm để các chuỗi về cùng độ dài
def pad_sequences(sequences, pad_id):
    max_len = max(len(seq) for seq in sequences)
    padded = []
    for seq in sequences:
        padded.append(seq + [pad_id] * (max_len - len(seq)))
    return torch.tensor(padded, dtype=torch.long)

# Build vocabularies
src_vocab = Vocab().build(train_src)
tgt_vocab = Vocab().build(train_tgt)
# Encode training data
train_src_ids = [src_vocab.encode(sent) for sent in train_src]
train_tgt_ids = [tgt_vocab.encode(sent) for sent in train_tgt]

# Sử dụng hàm pad_sequences để gán các token đệm
PAD_ID = src_vocab.word2id["<PAD>"]
# Lấy 3 câu đầu tiên để minh họa
sample_src_ids = train_src_ids[:3]
print("Trước khi padding:")
for i, seq in enumerate(sample_src_ids):
    print(f"  Câu {i+1}: {seq} (độ dài: {len(seq)})")

# Áp dụng padding
padded_sample = pad_sequences(sample_src_ids, PAD_ID)
print(f"\nSau khi padding (PAD_ID={PAD_ID}):")
print(padded_sample)
print(f"Shape: {padded_sample.shape}")



Trước khi padding:
  Câu 1: [1, 4, 5, 6, 7, 2] (độ dài: 6)
  Câu 2: [1, 4, 5, 6, 8, 2] (độ dài: 6)
  Câu 3: [1, 9, 10, 11, 2] (độ dài: 5)

Sau khi padding (PAD_ID=0):
tensor([[ 1,  4,  5,  6,  7,  2],
        [ 1,  4,  5,  6,  8,  2],
        [ 1,  9, 10, 11,  2,  0]])
Shape: torch.Size([3, 6])


## 4. Implement Encoder


In [10]:
class Encoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden_dim: int, pad_id: int = 0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)

    def forward(self, x: torch.Tensor):
        emb = self.embedding(x)
        outputs, hidden = self.rnn(emb)
        return outputs, hidden

## 5. Implement Decoder


In [11]:
class Decoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden_dim: int, pad_id: int = 0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token: torch.Tensor, hidden: torch.Tensor):
        emb = self.embedding(token)
        output, hidden = self.rnn(emb, hidden)
        logits = self.fc(output)
        return logits, hidden


## 6. Implement Seq2Seq Model with Teacher Forcing


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, bos_id: int, eos_id: int, device: str = "cpu"):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.device = device

    def forward(self, src: torch.Tensor, tgt: torch.Tensor, teacher_forcing_ratio: float = 0.5):
        batch_size, tgt_len = tgt.size()
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, tgt_len, vocab_size, device=src.device)

        _, hidden = self.encoder(src)
        decoder_input = tgt[:, 0].unsqueeze(1)

        for t in range(1, tgt_len):
            logits, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t:t+1, :] = logits

            top1 = logits.argmax(dim=-1)
            use_teacher = random.random() < teacher_forcing_ratio
            decoder_input = tgt[:, t].unsqueeze(1) if use_teacher else top1

        return outputs

    @torch.no_grad()
    def greedy_decode(self, src: torch.Tensor, max_len: int = 20):
        batch_size = src.size(0)
        _, hidden = self.encoder(src)
        decoder_input = torch.full((batch_size, 1), self.bos_id, dtype=torch.long, device=src.device)

        predictions = [[self.bos_id] for _ in range(batch_size)]

        for _ in range(max_len):
            logits, hidden = self.decoder(decoder_input, hidden)
            next_token = logits.argmax(dim=-1)
            decoder_input = next_token

            for i in range(batch_size):
                token_id = int(next_token[i, 0].item())
                predictions[i].append(token_id)

        return predictions


## 7. Train the model

In [5]:
# Suggested hyperparameters
EMB_DIM = 64
HIDDEN_DIM = 128
EPOCHS = 100
LR = 1e-3

# Create model using imported classes
PAD_ID = src_vocab.word2id["<PAD>"]
BOS_ID = tgt_vocab.word2id["<BOS>"]
EOS_ID = tgt_vocab.word2id["<EOS>"]

encoder = Encoder(len(src_vocab.word2id), EMB_DIM, HIDDEN_DIM, pad_id=PAD_ID)
decoder = Decoder(len(tgt_vocab.word2id), EMB_DIM, HIDDEN_DIM, pad_id=PAD_ID)
model = Seq2Seq(encoder, decoder, BOS_ID, EOS_ID, device=DEVICE).to(DEVICE)

print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

# Define optimizer and loss
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

# Training loop
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    
    # Simple batch training (process all data as one batch for simplicity)
    src_tensor = pad_sequences(train_src_ids, PAD_ID).to(DEVICE)
    tgt_tensor = pad_sequences(train_tgt_ids, PAD_ID).to(DEVICE)
    
    optimizer.zero_grad()
    
    # Forward pass
    output = model(src_tensor, tgt_tensor, teacher_forcing_ratio=0.5)
    
    # Compute loss (ignore first token which is <BOS>)
    output_dim = output.shape[-1]
    output = output[:, 1:, :].reshape(-1, output_dim)
    tgt = tgt_tensor[:, 1:].reshape(-1)
    
    loss = criterion(output, tgt)
    
    # Backward pass
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    total_loss = loss.item()
    
    # Print progress
    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{EPOCHS}, Loss: {total_loss:.4f}")

print("\nTraining completed!")


Model created with 163835 parameters
Epoch 10/100, Loss: 3.7242
Epoch 20/100, Loss: 3.1668
Epoch 30/100, Loss: 2.6908
Epoch 40/100, Loss: 2.1014
Epoch 50/100, Loss: 1.7875
Epoch 60/100, Loss: 1.2419
Epoch 70/100, Loss: 1.1451
Epoch 80/100, Loss: 0.6942
Epoch 90/100, Loss: 0.5316
Epoch 100/100, Loss: 0.3581

Training completed!


## 8. Evaluate on the dev set

In [6]:
# Encode dev set
dev_src_ids = [src_vocab.encode(sent) for sent in dev_src]
dev_src_tensor = pad_sequences(dev_src_ids, PAD_ID).to(DEVICE)

# Run greedy decoding
model.eval()
predictions = model.greedy_decode(dev_src_tensor, max_len=20)

# Print source / reference / prediction
print("=" * 80)
print("EVALUATION ON DEV SET")
print("=" * 80)

for i in range(len(dev_src)):
    src_text = dev_src[i]
    ref_text = dev_tgt[i]
    pred_ids = predictions[i]
    pred_text = tgt_vocab.decode(pred_ids)
    
    print(f"\n[Example {i+1}]")
    print(f"Source:     {src_text}")
    print(f"Reference:  {ref_text}")
    print(f"Prediction: {pred_text}")
    print("-" * 80)


EVALUATION ON DEV SET

[Example 1]
Source:     i am a teacher
Reference:  tôi là giáo viên
Prediction: tôi là giáo viên
--------------------------------------------------------------------------------

[Example 2]
Source:     she likes music
Reference:  cô ấy thích âm nhạc
Prediction: cô ấy thích âm nhạc
--------------------------------------------------------------------------------

[Example 3]
Source:     this is a book
Reference:  đây là một cuốn sách
Prediction: đây là một cuốn sách
--------------------------------------------------------------------------------

[Example 4]
Source:     we study machine learning
Reference:  chúng tôi học máy học
Prediction: chúng tôi học máy học
--------------------------------------------------------------------------------


## 9. Conceptual questions
1. What is teacher forcing?
2. Why can a basic Seq2Seq model struggle with long sentences?
3. What distribution does the decoder model at time step t?